# 04 - The VRP signal and the forecast test

Milestone 4 of `vol-strategy-backtester` adds a signal and four strategies that trade it. The signal is the log ratio of the implied variance over the next 30 days to a forecast of the realised variance over the same days. This notebook copies the pieces that compute it out of `src/volbt/signals/` (`realised.py`, `har.py`, `vrp.py` and `regime.py`) and the paired-loss inference of the forecast test out of `src/volbt/analytics/inference.py`. It runs them on **made-up data** from a seeded generator: no market data, no store and no `volbt`. Nothing here is a result.

This notebook is generated by `tools/build_notebooks.py`. Do not edit it by hand; edit `src/` and rebuild. The walkthrough, with one traced position, is in `docs/milestones/04-strategies-and-signal.md`.

## Realised variance from daily bars

A return is `ln((C_i + D_i) / C_{i-1})`: the unadjusted close plus the dividend whose ex-date is day i, over the last close. Adding the dividend back means an ex-date's price drop is not counted as variance. A weekend is one return. Copied from `src/volbt/signals/realised.py`.

In [ ]:
import numpy as np

In [ ]:
def as_days(values) -> np.ndarray:
    """Dates from a view's frame (Timestamps or dates) as numpy datetime64[D]."""
    return np.asarray(values).astype("datetime64[D]")

In [ ]:
def total_log_returns(bars, dividends) -> tuple[np.ndarray, np.ndarray]:
    """(dates, r) for every bar after the first: dates as datetime64[D], r_i = ln((C_i + D_i) / C_{i-1}).

    `bars` has `date` and `close` (view.spy_bars()), sorted by date; `dividends` has `ex_date` and `amount`
    (view.dividends()).
    """
    days = as_days(bars["date"])
    close = np.asarray(bars["close"], dtype=float)
    added = np.zeros(close.size)
    if len(dividends):
        ex = as_days(dividends["ex_date"])
        amount = np.asarray(dividends["amount"], dtype=float)
        at = np.searchsorted(days, ex, "left")          # the ex-date's bar, or the first bar after it
        keep = (at > 0) & (at < days.size)
        np.add.at(added, at[keep], amount[keep])
    if close.size < 2:
        return days[1:], np.zeros(0)
    return days[1:], np.log((close[1:] + added[1:]) / close[:-1])

In [ ]:
def window_variance(r, n: int) -> np.ndarray:
    """Element i: the mean of r^2 over r[i-n+1 .. i]; NaN for i < n - 1."""
    return window_mean(np.asarray(r, dtype=float) ** 2, n)

In [ ]:
def window_mean(v, n: int) -> np.ndarray:
    """Element i: the mean of v[i-n+1 .. i]; NaN for i < n - 1.

    A direct sum per window (np.convolve), so each element depends only on its own n values: the same element is
    bit-identical whatever comes after it (a cumulative sum would not be).
    """
    v = np.asarray(v, dtype=float)
    out = np.full(v.size, np.nan)
    if n >= 1 and v.size >= n:
        out[n - 1:] = np.convolve(v, np.ones(n), "valid") / n
    return out

In [ ]:
# Six made-up closes. Day 3 (index 3) is an ex-date: the close falls by the 0.50 dividend.
days = np.array(['2024-03-01', '2024-03-04', '2024-03-05', '2024-03-06', '2024-03-07', '2024-03-08'],
                dtype='datetime64[D]')
close = np.array([100.0, 101.0, 100.5, 100.0, 100.5, 101.5])
bars = {'date': days, 'close': close}
ex_dates = {'ex_date': days[3:4], 'amount': np.array([0.50])}
_, r_plain = total_log_returns(bars, {'ex_date': days[:0], 'amount': np.array([])})
_, r_total = total_log_returns(bars, ex_dates)
print('ex-date return without the dividend:', round(float(r_plain[2]), 5))
print('ex-date return with the dividend:   ', round(float(r_total[2]), 5))
print('mean of r^2 over 3 returns:', np.round(window_variance(r_total, 3), 8))

## HAR-RV, fitted only on what had ended

The regressors are the day's squared return, its 5-day mean and its 22-day mean; the target is the mean squared return of the next 21 sessions. A fit dated F uses only the pairs whose target ends on or before F, so no later data reaches it. Copied from `src/volbt/signals/har.py`, without `HarFit` and `HarForecaster` (classes). The first cell redefines `HarFit`, because the functions name it in their annotations; `HarForecaster` is the engine's monthly refit state. The two flag constants come from `volbt.signals.base`; `tests/test_notebooks.py` checks that the constants and the `HarFit` fields equal the real ones.

In [ ]:
from dataclasses import dataclass
from datetime import date

RV_FALLBACK = 1 << 0   # volbt.signals.base: the forecast was <= 0, the naive one stands in
RV_MISSING = 1 << 1    # volbt.signals.base: no forecast


@dataclass(frozen=True)
class HarFit:   # volbt.signals.har.HarFit; the copied functions below name it in their annotations
    fit_date: date
    coef: tuple[float, float, float, float]     # constant, day, week, month
    n_pairs: int

In [ ]:
from dataclasses import dataclass
from datetime import date

import numpy as np

In [ ]:
HORIZON = 21
MIN_PAIRS = 504
WEEK, MONTH = 5, 22

In [ ]:
def as_date(day) -> date:
    """A numpy datetime64 (or a date) as a plain date."""
    day = np.datetime64(day, "D").astype(object)
    return date(day.year, day.month, day.day)

In [ ]:
def regressors(v) -> np.ndarray:
    """The rows x_i = [1, v_i, mean v_{i-4..i}, mean v_{i-21..i}] for every i; NaN where the month is not complete."""
    v = np.asarray(v, dtype=float)
    x = np.column_stack([np.ones(v.size), v, window_mean(v, WEEK), window_mean(v, MONTH)])
    x[:MONTH - 1] = np.nan
    return x

In [ ]:
def targets(v, horizon: int = HORIZON) -> np.ndarray:
    """y_i = mean v_{i+1..i+horizon}; NaN where the window runs past the data."""
    v = np.asarray(v, dtype=float)
    y = np.full(v.size, np.nan)
    if v.size > horizon:
        y[:v.size - horizon] = window_mean(v, horizon)[horizon:]
    return y

In [ ]:
def fit(v, dates, fit_date: date, horizon: int = HORIZON, min_pairs: int = MIN_PAIRS) -> HarFit | None:
    """OLS on the pairs (x_i, y_i) whose target ends on or before fit_date; None with fewer than min_pairs.

    `v` and `dates` (datetime64[D] or dates) are aligned and ascending. Only v up to fit_date enters the fit.
    """
    days = np.asarray(dates).astype("datetime64[D]")
    end = int(np.searchsorted(days, np.datetime64(fit_date, "D"), "right"))   # v[:end] is dated <= fit_date
    v = np.asarray(v, dtype=float)[:end]
    x, y = regressors(v), targets(v, horizon)
    rows = np.flatnonzero(np.isfinite(x).all(axis=1) & np.isfinite(y))
    if rows.size < min_pairs:
        return None
    coef, *_ = np.linalg.lstsq(x[rows], y[rows], rcond=None)
    return HarFit(fit_date, tuple(float(c) for c in coef), int(rows.size))

In [ ]:
def naive_rv22(v) -> float:
    """mean v_{t-21..t} (22 squared returns); NaN with fewer."""
    month = window_mean(v, MONTH)
    return float(month[-1]) if month.size else float("nan")

In [ ]:
def har_forecast(har: HarFit, v) -> tuple[float, int]:
    """(y^_t, flags): x_t . coef at the last index of v; a forecast <= 0 or not finite gives the naive RV22 and
    RV_FALLBACK (plan section 4)."""
    x = regressors(v)
    if not x.shape[0]:
        return float("nan"), RV_MISSING
    forecast = float(x[-1] @ np.asarray(har.coef))
    if np.isfinite(forecast) and forecast > 0:
        return forecast, 0
    naive = naive_rv22(v)
    return naive, RV_FALLBACK | (0 if np.isfinite(naive) else RV_MISSING)

In [ ]:
def month_start(dates, t) -> date:
    """The first of `dates` in t's calendar month (t itself when it is the first)."""
    days = np.asarray(dates).astype("datetime64[D]")
    first = np.datetime64(f"{t.year:04d}-{t.month:02d}-01", "D")
    return as_date(days[int(np.searchsorted(days, first, "left"))])

In [ ]:
# A seeded, made-up path: the log variance is a persistent AR(1), so quiet and wild spells cluster.
rng = np.random.default_rng(20261117)
n_sessions = 1600
log_var = np.zeros(n_sessions)
for i in range(1, n_sessions):
    log_var[i] = 0.97 * log_var[i - 1] + 0.25 * rng.standard_normal()
r = 0.009 * np.exp(log_var / 2) * rng.standard_normal(n_sessions)
weekdays = np.arange('2015-01-01', '2024-01-01', dtype='datetime64[D]')
dates = weekdays[np.is_busday(weekdays)][:n_sessions]
v = r ** 2
t0 = 1100                                    # the session whose forecast we want
fit_date = as_date(dates[t0 - 10])           # a fit dated ten sessions before t0
fit_a = fit(v, dates, fit_date)
print('fit dated', fit_a.fit_date, 'uses', fit_a.n_pairs, 'pairs; coef', np.round(fit_a.coef, 6))
# Change every value after t0: multiply by 3 and permute. The fit and the forecast at t0 must not move.
v_later = v.copy()
v_later[t0 + 1:] = rng.permutation(v[t0 + 1:] * 3)
fit_b = fit(v_later, dates, fit_date)
forecast_a, flags_a = har_forecast(fit_a, v[:t0 + 1])
forecast_b, flags_b = har_forecast(fit_b, v_later[:t0 + 1])
print('fit unchanged by the future:', fit_a == fit_b)
print('forecast unchanged by the future:', forecast_a == forecast_b, 'flags', flags_a)
# The mutant: one fit on the whole series (a full-sample fit) is not causal.
full = fit(v, dates, as_date(dates[-1]))
full_later = fit(v_later, dates, as_date(dates[-1]))
print('a full-sample fit moves when the future moves:', full != full_later)
print('naive RV22 at t0:', round(naive_rv22(v[:t0 + 1]), 8), 'HAR forecast:', round(forecast_a, 8))
print('month_start:', month_start(dates[:t0 + 1], as_date(dates[t0])))

## The log variance ratio, and why the window must match

At session t the signal is `s = ln(IV30^2 / RV^2)`, where `RV^ = sqrt(n y^ 365 / 30)` and n is the number of sessions in the next 30 calendar days. A window holds 17 to 22 sessions, not a fixed 21, so `n` comes from the calendar. Copied from `src/volbt/signals/vrp.py`, without `VrpSignal` (the class the engine owns). `DAYS` and `DAYS_PER_YEAR` come from `volbt.signals.iv30`, which needs the pricing code; `tests/test_notebooks.py` checks them.

In [ ]:
DAYS = 30             # volbt.signals.iv30
DAYS_PER_YEAR = 365   # volbt.signals.iv30

In [ ]:
import math
from typing import Any

In [ ]:
RV_MODELS = ("har", "naive")

In [ ]:
def rv_hat(n_window: int, forecast: float) -> float:
    """sqrt(n y^ 365 / 30); NaN unless the forecast is finite and positive and the window holds a session."""
    if not (math.isfinite(forecast) and forecast > 0 and n_window > 0):
        return float("nan")
    return math.sqrt(n_window * forecast * DAYS_PER_YEAR / DAYS)

In [ ]:
# Made-up numbers: IV30 15.2%, a forecast mean daily variance of 8.4e-5.
iv30, y_hat = 0.152, 8.4e-5
for n_window in (22, 21, 17):
    rv = rv_hat(n_window, y_hat)
    print(f'n = {n_window}: RV^ = {rv:.4f}, s = {2 * math.log(iv30 / rv):+.4f}')
# Annualising with 252 sessions treats every window as 21 sessions. On a 17-session window that is
# a bias of ln(21/17) in the ratio:
print('ln(21/17) =', round(math.log(21 / 17), 4))
print(rv_hat(0, y_hat), rv_hat(21, -1.0))   # no session in the window, or a forecast <= 0: NaN

## VIX regimes, lagged

A strategy may use a volatility regime only lagged: at t it sees the label of the previous VIX session. Copied from `src/volbt/signals/regime.py`.

In [ ]:
from datetime import date

import numpy as np

In [ ]:
BOUNDS = (15.0, 20.0, 30.0)
BUCKETS = ("<15", "15-20", "20-30", ">=30")
SHOCK_RISE = 0.25            # +25% close to close
SHOCK_POINTS = 5.0           # or +5 points
POST_SHOCK = 21              # the sessions after a shock that form the post-shock regime

In [ ]:
def vix_bucket(level: float) -> str:
    """The ADR-0010 bucket of one VIX close: 19.99 -> "15-20", 20.00 -> "20-30"."""
    return BUCKETS[int(np.searchsorted(BOUNDS, level, "right"))]

In [ ]:
def shock_days(closes) -> np.ndarray:
    """Element i: VIX rose at least +25% or at least +5 points from close i-1 to close i (False at i = 0)."""
    closes = np.asarray(closes, dtype=float)
    out = np.zeros(closes.size, dtype=bool)
    if closes.size > 1:
        before, after = closes[:-1], closes[1:]
        out[1:] = (after >= before * (1 + SHOCK_RISE)) | (after - before >= SHOCK_POINTS)
    return out

In [ ]:
def regime_labels(closes) -> list[str | None]:
    """Element i: "shock", "post_shock" (a shock among the POST_SHOCK sessions before i) or "calm"; None at i = 0."""
    shocks = shock_days(closes)
    labels: list[str | None] = []
    for i in range(shocks.size):
        if i == 0:
            labels.append(None)
        elif shocks[i]:
            labels.append("shock")
        elif shocks[max(i - POST_SHOCK, 0):i].any():
            labels.append("post_shock")
        else:
            labels.append("calm")
    return labels

In [ ]:
def lagged_regime(vix, t: date) -> tuple[str | None, str | None]:
    """(bucket, regime) of the last VIX session dated before t, from view.vix() (`date`, `close`)."""
    days = np.asarray(vix["date"]).astype("datetime64[D]")
    end = int(np.searchsorted(days, np.datetime64(t, "D"), "left"))     # rows dated before t
    if end == 0:
        return None, None
    # The last row's label needs the shocks of the POST_SHOCK rows before it, and each shock the close before it.
    closes = np.asarray(vix["close"], dtype=float)[max(end - POST_SHOCK - 2, 0):end]
    return vix_bucket(float(closes[-1])), regime_labels(closes)[-1]

In [ ]:
# Made-up VIX closes: day 2 rises 27% (a shock), and the next sessions are post-shock.
vix_closes = np.array([16.0, 16.5, 21.0, 20.0, 19.0])
print(vix_bucket(19.99), vix_bucket(20.0))
print(shock_days(vix_closes))
print(regime_labels(vix_closes))
vix = {'date': np.array(['2024-03-01', '2024-03-04', '2024-03-05', '2024-03-06', '2024-03-07'],
                        dtype='datetime64[D]'), 'close': vix_closes}
print(lagged_regime(vix, date(2024, 3, 6)), lagged_regime(vix, date(2024, 3, 1)))

## The forecast test: a paired loss and a block bootstrap

`d` is the paired difference QLIKE(naive) - QLIKE(HAR) on each session, so `d > 0` favours HAR. Neighbouring sessions overlap in their 21-day targets, so `d` is strongly autocorrelated, and the test must keep that dependence: an equal-weighted-cosine (EWC) t with a moving-block bootstrap. QLIKE is a proxy loss (the realised variance of daily closes stands in for the true variance). Copied from `src/volbt/analytics/inference.py`.

In [ ]:
import math

import numpy as np

In [ ]:
BLOCKS = (21, 42, 84, 168)
CHUNK_CELLS = 1 << 20          # values drawn per chunk: about 8 MB of float64
TIE = 1e-12

In [ ]:
def series(d) -> np.ndarray:
    """d as a 1-D float array; ValueError if it is empty, not 1-D or not finite."""
    d = np.asarray(d, dtype=float)
    if d.ndim != 1 or d.size == 0:
        raise ValueError(f"d must be a non-empty 1-D array, got shape {d.shape}")
    if not np.isfinite(d).all():
        raise ValueError("d must be finite (a NaN would decide every comparison of a sign-flip draw)")
    return d

In [ ]:
def check_block(block: int, n: int) -> int:
    """block as an int; ValueError unless 1 <= block <= n."""
    if not 1 <= int(block) <= n:
        raise ValueError(f"block must be in 1..{n}, got {block}")
    return int(block)

In [ ]:
def qlike(y, yhat) -> np.ndarray:
    """y/yhat - ln(y/yhat) - 1, elementwise (Patton 2011). y and yhat > 0; numpy's NaN or inf otherwise.

    The loss of forecasting the variance yhat when the proxy is y. It is >= 0 and 0 only at yhat == y, and its expected
    value given a proxy with E[y] = h is smallest at yhat = h: so it ranks forecasts with a noisy unbiased proxy as it
    would with the true variance (tests/test_inference.py, E3). It is asymmetric: an under-forecast costs more than
    an over-forecast by the same factor (y = 2, yhat = 1 gives 0.3069; y = 1, yhat = 2 gives 0.1931), so the forecast
    closer to y in absolute terms does not always have the lower loss.
    """
    ratio = np.asarray(y, dtype=float) / np.asarray(yhat, dtype=float)
    return ratio - np.log(ratio) - 1.0

In [ ]:
def autocov(x: np.ndarray, nlags: int) -> np.ndarray:
    """gamma_j = sum_t (x_t - mean)(x_{t-j} - mean) / n for j = 0..nlags (0 at j >= n)."""
    if nlags < 0:
        raise ValueError(f"nlags must be >= 0, got {nlags}")
    x = x - x.mean()
    n = x.size
    return np.array([np.dot(x[j:], x[:n - j]) / n if j < n else 0.0 for j in range(nlags + 1)])

In [ ]:
def acf(x, nlags: int) -> np.ndarray:
    """Autocorrelations at lags 0..nlags: gamma_j / gamma_0, the mean removed once (lag 0 is 1).

    [1, 2, 3, 4] gives [1, 0.25, -0.3, -0.45]. A constant x (gamma_0 = 0) gives NaN at every lag.
    """
    gamma = autocov(series(x), int(nlags))
    if gamma[0] == 0:
        return np.full(gamma.size, np.nan)
    return gamma / gamma[0]

In [ ]:
def mb_se(d, block: int) -> float:
    """The moving-block bootstrap SE of mean(d): sqrt(var(B_j, ddof=0) / (n // block)), B_j the mean of d[j : j+block]
    (whole blocks only, floor(n / block) of them: the module docstring says why the CI's count differs).

    By hand, d = [1, 2, 3, 4], block 2: B = [1.5, 2.5, 3.5], var 2/3, k = 2, so se = sqrt(1/3) = 0.57735. Block 1 gives
    the iid bootstrap SE, std(d, ddof=0) / sqrt(n).
    """
    d = series(d)
    b = check_block(block, d.size)
    means = np.convolve(d, np.ones(b), "valid") / b       # B_j for j = 0..n-b, each its own sum
    return math.sqrt(float(np.var(means)) / (d.size // b))

In [ ]:
def block_sweep(d, blocks=BLOCKS, tol: float = 0.10, minimum: int = 42) -> dict:
    """The SE at each block and the block the rule picks (plan section 6).

    Rule: the smallest b in `blocks` with b >= minimum, 2b also in `blocks` and |se(2b)/se(b) - 1| < tol (with the
    default list the candidates are 42 and 84). If none qualifies, the SE still grows at every doubling, so the
    dependence reaches past every block the rule can pick: then the largest block of the list (168), the widest SE
    and the fewest blocks, with converged False (the plan does not say; default taken 2026-11-09, owner can
    override). A ratio whose se(b) is 0 is NaN and never qualifies.

    Returns {"se": {b: se}, "ratios": {b: se(2b)/se(b)} for every b whose 2b is listed, "block": int,
    "converged": bool, "rule": the rule in one line}, in plain Python ints and floats.
    """
    d = series(d)
    blocks = sorted(int(b) for b in blocks)
    paired = [b for b in blocks if 2 * b in blocks]
    if not paired:
        raise ValueError(f"no block in {blocks} has its double in the list")
    se = {b: float(mb_se(d, b)) for b in blocks}
    ratios = {b: se[2 * b] / se[b] if se[b] > 0 else float("nan") for b in paired}
    passing = [b for b in paired if b >= minimum and abs(ratios[b] - 1) < tol]
    chosen = passing[0] if passing else blocks[-1]
    rule = (f"smallest b in {blocks} with b >= {minimum}, 2b listed and |se(2b)/se(b) - 1| < {tol}; "
            f"else the largest b of the list, converged False")
    return {"se": se, "ratios": ratios, "block": int(chosen), "converged": bool(passing), "rule": rule}

In [ ]:
SUMMARY_LAGS = (1, 5, 21, 63, 126, 252)
SUMMARY_KEYS = tuple(f"acf{lag}" for lag in SUMMARY_LAGS) + ("r21", "r42", "r84", "skew", "kurt", "spec4_20")

In [ ]:
def dependence_summaries(d) -> dict[str, float]:
    """Twelve centred summaries of d (fix pass 2, item 9; review A's summ.py): the ACF at lags 1, 5, 21, 63, 126 and
    252; the SE ratios mb_se(2b) / mb_se(b) at b = 21, 42, 84; the sample skewness and excess kurtosis (moments of
    (d - mean) / std, ddof 0); and spec4_20, the mean squared projection of d on its 4 lowest cosines over that on
    cosines 5..20. Each is invariant to adding a constant to d, and none but the skewness changes sign with d: so none
    reveals mean(d) or its sign (the skewness is the third moment about the mean). n must exceed 336 (2 x 168).

    Returns {key: float} for SUMMARY_KEYS, in that order.
    """
    d = series(d)
    if d.size <= 2 * BLOCKS[-1]:
        raise ValueError(f"dependence_summaries needs more than {2 * BLOCKS[-1]} values, got {d.size}")
    rho = acf(d, SUMMARY_LAGS[-1])
    se = {b: mb_se(d, b) for b in BLOCKS}
    z = (d - d.mean()) / d.std()
    y = cosines(d.size, 20) @ d
    y2 = y * y
    out = {f"acf{lag}": float(rho[lag]) for lag in SUMMARY_LAGS}
    out.update({f"r{b}": se[2 * b] / se[b] for b in (21, 42, 84)})
    out["skew"] = float(np.mean(z ** 3))
    out["kurt"] = float(np.mean(z ** 4) - 3.0)
    out["spec4_20"] = float(y2[:4].mean() / y2[4:20].mean())
    return out

In [ ]:
def cosines(n: int, q: int) -> np.ndarray:
    """The (q, n) matrix sqrt(2/n) cos(pi j (t - 1/2) / n), j = 1..q, t = 1..n: orthonormal rows, each orthogonal to a
    constant, so the EWC variance ignores the mean."""
    if not 1 <= int(q) < n:
        raise ValueError(f"q must be in 1..{n - 1}, got {q}")
    t = (np.arange(1, n + 1) - 0.5) / n
    return math.sqrt(2.0 / n) * np.cos(np.pi * np.outer(np.arange(1, int(q) + 1), t))

In [ ]:
def _ewc_rows(x: np.ndarray, weights: np.ndarray) -> np.ndarray:
    """The EWC t of each row of x (rows, n) with the cosines `weights` (q, n)."""
    y = x @ weights.T
    lrv = (y * y).mean(axis=1)
    n = x.shape[1]
    with np.errstate(divide="ignore", invalid="ignore"):
        return math.sqrt(n) * x.mean(axis=1) / np.sqrt(lrv)

In [ ]:
def ewc_t(d, q: int) -> float:
    """sqrt(n) mean(d) / sqrt(mean_j Y_j^2), Y_j = sqrt(2/n) sum_t cos(pi j (t - 1/2) / n) d_t (module docstring).

    t_q under iid normal d. By hand, d = [1, 2, 3, 4], q = 1: Y_1 = sqrt(1/2) (cos(pi/8) + 2 cos(3pi/8) - 3 cos(3pi/8)
    - 4 cos(pi/8)) = -2.230442, so t = 2 x 2.5 / 2.230442 = 2.2417076. A d whose cosine projections are all 0 (a
    constant) gives +-inf, or NaN at a zero mean.
    """
    d = series(d)
    return float(_ewc_rows(d[None, :], cosines(d.size, q))[0])

In [ ]:
def ewc_bootstrap(d, q: int, block: int, draws: int = 9_999, *, seed, level: float = 0.95) -> dict:
    """The forecast test (module docstring): {"t", "se", "p", "ci", "draws", "q", "block"}.

    t is ewc_t(d, q); se = sqrt(mean_j Y_j^2 / n); p the one-sided moving-block bootstrap-t p-value of mean(d) <= 0
    against mean(d) > 0, (1 + #{t* >= t}) / (1 + draws); ci = (mean - t*_hi se, mean - t*_lo se), t*_lo and t*_hi the
    (1 - level)/2 and (1 + level)/2 quantiles of t* (numpy's linear quantile). d identically 0 has no EWC t (0/0):
    ValueError. The draws come in chunks from rng.random(), so for a seed p is chunk-independent and the CI is to
    rounding (module docstring, "Randomness").
    """
    d = series(d)
    n = d.size
    b = check_block(block, n)
    if b == n:
        raise ValueError("block must be shorter than d")
    if draws < 1:
        raise ValueError(f"draws must be >= 1, got {draws}")
    if not 0 < level < 1:
        raise ValueError(f"level must be in (0, 1), got {level}")
    weights = cosines(n, q)
    y = weights @ d
    lrv = float(np.mean(y * y))
    if lrv <= 0:
        raise ValueError("d has no low-frequency variation (a constant): the EWC t is not defined")
    se = math.sqrt(lrv / n)
    t = float(d.mean()) / se
    k = -(-n // b)
    starts_n = n - b + 1
    centre = float(np.convolve(d, np.ones(b), "valid").mean()) / b    # ~ E* mean* (exact but for the cut last block)
    offsets = np.arange(b)
    rng = np.random.default_rng(seed)
    stars = []
    for rows in chunks(draws, k * b):
        starts = (rng.random((rows, k)) * starts_n).astype(np.intp)
        resample = d[(starts[:, :, None] + offsets).reshape(rows, k * b)[:, :n]]
        stars.append(_ewc_rows(resample - centre, weights))
    stars = np.concatenate(stars)
    p = (1 + int(np.count_nonzero(stars >= t))) / (1 + draws)
    lo, hi = np.quantile(stars, [(1 - level) / 2, (1 + level) / 2])
    mean = float(d.mean())
    return {"t": t, "se": se, "p": p, "ci": (mean - float(hi) * se, mean - float(lo) * se), "draws": int(draws),
            "q": int(q), "block": b}

In [ ]:
def chunks(draws: int, width: int):
    """Row counts that add up to `draws`, each chunk about CHUNK_CELLS values of `width` columns."""
    rows = max(1, CHUNK_CELLS // max(width, 1))
    for start in range(0, draws, rows):
        yield min(rows, draws - start)

In [ ]:
def block_signflip(d, block: int, draws: int = 10_000, *, seed, alternative: str = "greater") -> float:
    """The block sign-flip p-value (module docstring): "greater" tests mean(d) <= 0 against mean(d) > 0.

    "less" counts draws at or below the observed mean (the same as "greater" on -d), "two-sided" draws whose |mean*|
    is at or above |mean_obs|. 10,000 draws on n = 2,495 take about 0.15 s at block 1 and a few ms at block 42.
    """
    d = series(d)
    b = check_block(block, d.size)
    if alternative not in ("greater", "less", "two-sided"):
        raise ValueError(f"alternative must be 'greater', 'less' or 'two-sided', got {alternative!r}")
    if draws < 1:
        raise ValueError(f"draws must be >= 1, got {draws}")
    n = d.size
    sums = np.add.reduceat(d, np.arange(0, n, b))           # block sums; the last block holds the remainder
    observed = float(sums.sum()) / n
    tie = TIE * float(np.abs(sums).sum()) / n
    rng = np.random.default_rng(seed)
    count = 0
    for rows in chunks(draws, sums.size):
        means = np.where(rng.random((rows, sums.size)) < 0.5, -1.0, 1.0) @ sums / n
        if alternative == "greater":
            count += int(np.count_nonzero(means >= observed - tie))
        elif alternative == "less":
            count += int(np.count_nonzero(means <= observed + tie))
        else:
            count += int(np.count_nonzero(np.abs(means) >= abs(observed) - tie))
    return (1 + count) / (1 + draws)

In [ ]:
def nw_t(d, lags: int) -> float:
    """mean(d) / sqrt(LRV / n), LRV = gamma_0 + 2 sum_{j=1..lags} (1 - j/(lags+1)) gamma_j (Newey-West, Bartlett).

    gamma_j is the biased autocovariance (divided by n), so lags = 0 gives the plain t with the ddof=0 std,
    mean / (std(d, ddof=0) / sqrt(n)). When LRV <= 0 (a constant d): 0.0 if the mean is 0, else +-inf with the mean's
    sign (default taken 2026-11-09, owner can override). By hand, d = [1, 2, 3, 4], lags 1: gamma = [1.25, 0.3125],
    LRV = 1.25 + 2 x 0.5 x 0.3125 = 1.5625, t = 2.5 / sqrt(1.5625 / 4) = 4.0.
    """
    d = series(d)
    lags = int(lags)
    gamma = autocov(d, lags)
    weights = 1.0 - np.arange(1, lags + 1) / (lags + 1)
    lrv = float(gamma[0] + 2.0 * np.dot(weights, gamma[1:]))
    mean = float(d.mean())
    if lrv <= 0:
        return 0.0 if mean == 0 else math.copysign(math.inf, mean)
    return mean / math.sqrt(lrv / d.size)

In [ ]:
def block_bootstrap_ci(d, block: int, draws: int = 10_000, *, seed, level: float = 0.95) -> tuple[float, float]:
    """The moving-block bootstrap percentile CI of mean(d).

    A resample is k = ceil(n / b) blocks d[s : s+b], each start s uniform on 0..n-b (with replacement), joined and cut
    to n values: the last block keeps its first r = n - (k-1) b values. The CI is the (1-level)/2 and (1+level)/2
    quantiles of the resample means, numpy's default ("linear") quantile. d identically 0 gives (0.0, 0.0).
    """
    d = series(d)
    b = check_block(block, d.size)
    if not 0 < level < 1:
        raise ValueError(f"level must be in (0, 1), got {level}")
    if draws < 1:
        raise ValueError(f"draws must be >= 1, got {draws}")
    n = d.size
    k = -(-n // b)
    r = n - (k - 1) * b
    full = np.convolve(d, np.ones(b), "valid")                     # the sum of d[s : s+b], s = 0..n-b
    last = np.convolve(d, np.ones(r), "valid")[:n - b + 1]         # the sum of d[s : s+r], same starts
    rng = np.random.default_rng(seed)
    means = []
    for rows in chunks(draws, k):
        starts = (rng.random((rows, k)) * (n - b + 1)).astype(np.intp)
        means.append((full[starts[:, :-1]].sum(axis=1) + last[starts[:, -1]]) / n)
    means = np.concatenate(means)
    lo, hi = np.quantile(means, [(1 - level) / 2, (1 + level) / 2])
    return float(lo), float(hi)

In [ ]:
def holm(pvalues) -> np.ndarray:
    """Holm step-down adjusted p-values, in the input order.

    Sorted ascending, p_(i) becomes max over j <= i of min(1, (m - j + 1) p_(j)): monotone in the sort and capped at 1.
    By hand, [0.01, 0.04, 0.03, 0.005] gives [0.03, 0.06, 0.06, 0.02].
    """
    p = np.asarray(pvalues, dtype=float)
    if p.ndim != 1:
        raise ValueError(f"pvalues must be 1-D, got shape {p.shape}")
    m = p.size
    order = np.argsort(p, kind="stable")
    stepped = np.maximum.accumulate(np.minimum(1.0, (m - np.arange(m)) * p[order]))
    adjusted = np.empty(m)
    adjusted[order] = stepped
    return adjusted

In [ ]:
# The hand-checkable cases from the docstrings.
print('qlike:', round(float(qlike(2.0, 1.0)), 4), round(float(qlike(1.0, 2.0)), 4))
print('acf:', np.round(acf([1, 2, 3, 4], 3), 4))
print('mb_se:', round(mb_se([1, 2, 3, 4], 2), 5), 'ewc_t:', round(ewc_t([1, 2, 3, 4], 1), 7))
# A made-up d: an AR(1) with rho 0.9 around a positive mean (0.25), 600 sessions.
gen = np.random.default_rng(7)
d = np.zeros(600)
for i in range(1, 600):
    d[i] = 0.9 * d[i - 1] + 0.1 * gen.standard_normal()
d = d + 0.25
test = ewc_bootstrap(d, 4, 21, draws=999, seed=1)
print('EWC t', round(test['t'], 3), 'p', test['p'], 'CI', tuple(round(x, 3) for x in test['ci']))
# The controls: -d must not reject, and a constant d has no EWC t.
print('p of -d:', ewc_bootstrap(-d, 4, 21, draws=999, seed=1)['p'])
try:
    ewc_bootstrap(np.zeros(600), 4, 21, draws=99, seed=1)
except ValueError as error:
    print('constant d:', error)
print('sign flip p:', block_signflip(d, 21, draws=999, seed=1), 'NW t:', round(nw_t(d, 21), 3))
print('Holm:', holm([0.01, 0.04, 0.03]))